# PIXELFORGE STUDIO — GPU Worker
شغّل Runtime بنوع T4 GPU، ثم نفّذ خلية التشغيل بالأسفل. تم تثبيت إصدارات متوافقة لتجنب خطأ FLAX_WEIGHTS_NAME. إذا شغّلت الخلية القديمة في الجلسة نفسها، اختر Runtime → Disconnect and delete runtime ثم افتح الدفتر مجدداً.
هذا العامل يستقبل مهام تعديل الصور من موقع PIXELFORGE ويرسل النتائج تلقائياً. اترك الجلسة مفتوحة أثناء الاستخدام.

In [ ]:
!pip -q install --upgrade --no-cache-dir "diffusers==0.32.2" "transformers==4.46.3" "tokenizers==0.20.3" accelerate safetensors pillow requests
import os, time, base64, io, getpass, requests, torch
from PIL import Image
import transformers, diffusers
print('Versions:', 'diffusers', diffusers.__version__, '| transformers', transformers.__version__)
from diffusers import StableDiffusionInstructPix2PixPipeline

API = 'https://pixelforge-studio-zajl.onrender.com'
WORKER_SECRET = getpass.getpass('PIXELFORGE worker secret (hidden): ')
if not WORKER_SECRET.strip(): raise ValueError('Worker secret is required')
HEADERS = {'Authorization': 'Bearer ' + WORKER_SECRET.strip()}
MODEL_ID = 'timbrooks/instruct-pix2pix'
print('Checking GPU...')
if not torch.cuda.is_available(): raise RuntimeError('GPU not enabled. In Colab choose Runtime > Change runtime type > T4 GPU.')
print('Loading image-editing model on GPU...')
pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False)
pipe.to('cuda')
pipe.enable_attention_slicing()
print('PIXELFORGE GPU worker is online. Keep this cell running.')
while True:
    try:
        r = requests.get(API + '/api/worker/next', headers=HEADERS, timeout=90)
        if r.status_code == 204:
            time.sleep(2); continue
        r.raise_for_status()
        job = r.json()['job']
        try:
            raw = job['image'].split(',',1)[1]
            img = Image.open(io.BytesIO(base64.b64decode(raw))).convert('RGB')
            img.thumbnail((768,768))
            prompt = str(job.get('prompt','')).strip()
            with torch.inference_mode():
                out = pipe(prompt=prompt, image=img, num_inference_steps=20, image_guidance_scale=1.5, guidance_scale=7.0).images[0]
            buf=io.BytesIO(); out.save(buf,format='JPEG',quality=92)
            encoded=base64.b64encode(buf.getvalue()).decode('ascii')
            payload={'id':job['id'],'image':'data:image/jpeg;base64,'+encoded}
        except Exception as e:
            payload={'id':job['id'],'error':str(e)[:450]}
        requests.post(API + '/api/worker/result', headers={**HEADERS,'Content-Type':'application/json'}, json=payload, timeout=120).raise_for_status()
        print('Completed job:', job['id'])
    except KeyboardInterrupt:
        print('Worker stopped.'); break
    except Exception as e:
        print('Worker connection/model error:', repr(e)); time.sleep(5)
